In [4]:
%pip install pyspark
from pyspark.sql import SparkSession

Note: you may need to restart the kernel to use updated packages.


In [ ]:
spark = SparkSession.builder.appName("E-Commerce").master("spark://localhost:7077").getOrCreate()
print(spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/09/14 12:23:48 WARN Utils: Your hostname, soc-5CG2243S91, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/09/14 12:23:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/14 12:23:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [42]:
from pyspark.sql.types import IntegerType, StringType, DateType, StructField, StructType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

products_data = [
    (1, "Laptop", "Electronics"), (2, "Keyboard", "Electronics"), (3, "Mouse", "Electronics"),
    (4, "T-Shirt", "Apparel"), (5, "Jeans", "Apparel"), (6, "Socks", "Apparel"),
    (7, "Coffee Maker", "Home Goods"), (8, "Blender", "Home Goods")
]
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False)
])
products_df = spark.createDataFrame(products_data, products_schema)

customers_data = [
    (101, "Alice", "2023-01-15"), (102, "Bob", "2023-02-20"),
    (103, "Charlie", "2023-03-05")
]
customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), False),
    StructField("join_date", StringType(), False)
])
customers_df = spark.createDataFrame(customers_data, customers_schema)

sales_data = [
    (1, 1, 101, 2, "2025-01-02 10:30:00"), (2, 4, 102, 3, "2025-01-05 11:00:00"),
    (3, 2, 101, 1, "2025-01-08 14:15:00"), (4, 7, 103, 1, "2025-02-10 08:00:00"),
    (5, 5, 102, None, "2025-02-15 18:45:00"), # <-- Note the null value
    (6, 1, 103, 1, "2025-02-20 12:00:00"), (7, 2, 102, 2, "2025-03-01 09:30:00"),
    (8, 8, 101, 1, "2025-03-05 16:20:00"), (9, 4, 103, 1, "2025-03-10 13:00:00")
]
sales_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("units_sold", IntegerType(), True),  # <-- Note the nullable field
    StructField("sale_timestamp", StringType(), False)
])
sales_df = spark.createDataFrame(sales_data, sales_schema)

print("DataFrames for the project have been created.")

DataFrames for the project have been created.


In [43]:
print(customers_df.printSchema())
print(products_df.printSchema())
print(sales_df.printSchema())

root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- join_date: string (nullable = false)

None
root
 |-- product_id: integer (nullable = false)
 |-- product_name: string (nullable = false)
 |-- category: string (nullable = false)

None
root
 |-- order_id: integer (nullable = false)
 |-- product_id: integer (nullable = false)
 |-- customer_id: integer (nullable = false)
 |-- units_sold: integer (nullable = true)
 |-- sale_timestamp: string (nullable = false)

None


In [44]:
sales_df = sales_df.fillna(1, subset=["units_sold"])
sales_df.show(10)

+--------+----------+-----------+----------+-------------------+
|order_id|product_id|customer_id|units_sold|     sale_timestamp|
+--------+----------+-----------+----------+-------------------+
|       1|         1|        101|         2|2025-01-02 10:30:00|
|       2|         4|        102|         3|2025-01-05 11:00:00|
|       3|         2|        101|         1|2025-01-08 14:15:00|
|       4|         7|        103|         1|2025-02-10 08:00:00|
|       5|         5|        102|         1|2025-02-15 18:45:00|
|       6|         1|        103|         1|2025-02-20 12:00:00|
|       7|         2|        102|         2|2025-03-01 09:30:00|
|       8|         8|        101|         1|2025-03-05 16:20:00|
|       9|         4|        103|         1|2025-03-10 13:00:00|
+--------+----------+-----------+----------+-------------------+



In [45]:
sales_df = sales_df.withColumn("sale_timestamp", F.to_timestamp(F.col("sale_timestamp"), "yyyy-MM-dd HH:mm:ss"))
customers_df = customers_df.withColumn(colName="join_date", col=F.to_date(F.col("join_date")))

In [46]:
enriched_sales_df = sales_df.join(products_df, on="product_id").join(customers_df, on="customer_id")

In [47]:
enriched_sales_df.printSchema()

root
 |-- customer_id: integer (nullable = false)
 |-- product_id: integer (nullable = false)
 |-- order_id: integer (nullable = false)
 |-- units_sold: integer (nullable = true)
 |-- sale_timestamp: timestamp (nullable = false)
 |-- product_name: string (nullable = false)
 |-- category: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- join_date: date (nullable = true)



In [48]:
enriched_sales_df.groupBy("product_id", "product_name").agg(F.sum(F.col("units_sold")).alias("Total Units Sold")).orderBy(F.desc(F.col("Total Units Sold"))).collect()

[Row(product_id=4, product_name='T-Shirt', Total Units Sold=4),
 Row(product_id=1, product_name='Laptop', Total Units Sold=3),
 Row(product_id=2, product_name='Keyboard', Total Units Sold=3),
 Row(product_id=8, product_name='Blender', Total Units Sold=1),
 Row(product_id=5, product_name='Jeans', Total Units Sold=1),
 Row(product_id=7, product_name='Coffee Maker', Total Units Sold=1)]

In [49]:
enriched_sales_df.withColumn("revenue", F.col("units_sold") * 25.5).groupBy("category").agg(F.sum(F.col("revenue")).alias("Total Revenue")).orderBy(F.desc("Total Revenue")).show(10)

+-----------+-------------+
|   category|Total Revenue|
+-----------+-------------+
|Electronics|        153.0|
|    Apparel|        127.5|
| Home Goods|         51.0|
+-----------+-------------+



In [ ]:
product_sales_df = enriched_sales_df.groupBy("category", "product_name").agg(F.sum(F.col("units_sold")).alias("Total Units Sold"))

window_spec = Window.partitionBy("category").orderBy(F.desc("Total Units Sold"))

product_sales_df.withColumn("rank", F.rank().over(window_spec)).filter(F.col("rank") == 1).show(10)


+-----------+------------+----------------+----+
|   category|product_name|Total Units Sold|rank|
+-----------+------------+----------------+----+
|    Apparel|     T-Shirt|               4|   1|
|Electronics|      Laptop|               3|   1|
|Electronics|    Keyboard|               3|   1|
| Home Goods|     Blender|               1|   1|
| Home Goods|Coffee Maker|               1|   1|
+-----------+------------+----------------+----+



25/09/15 07:12:16 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.3: worker lost: Not receiving heartbeat for 60 seconds
25/09/15 07:12:17 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
25/09/15 09:42:53 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.2: worker lost: Not receiving heartbeat for 60 seconds
25/09/15 09:43:00 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.3: worker lost: Not receiving heartbeat for 60 seconds
